# Machine Learning: Regressions

**Objectives:**
- Load and explore a dataset
- Split data into training and test sets
- Train regression models with scikit-learn (Linear, Lasso, Ridge)
- Evaluate model performance using $R^2$ scores
- Understand the purpose of cross-validation (k-fold)

We work through two datasets:  
1. **Diabetes dataset** — a small, pre-normalized dataset to learn the ML workflow  
2. **California Housing dataset** — a larger, real-world dataset to practice sparse regressions

---
**How this notebook works:**  
Setup cells (imports, data loading) are already filled in.  
You complete the cells marked with `# TODO`.

---
## Part 1: Diabetes Dataset — Basic Regression

### Step 1: Import and Explore the Data

We load the **diabetes dataset** from scikit-learn.  
It contains 10 baseline variables (age, sex, BMI, blood pressure, etc.) for 442 patients,  
and a target variable measuring disease progression one year later.

The dataset is returned as a dictionary with keys: `'data'`, `'target'`, `'feature_names'`, `'DESCR'`.

In [ ]:
# --- SETUP (run this cell) ---
import sklearn
import sklearn.datasets
import pandas
import seaborn
from matplotlib import pyplot as plt

dataset = sklearn.datasets.load_diabetes()

# The result is a dictionary:
# 'data'          -> features (X)
# 'target'        -> labels (y)
# 'feature_names' -> names of the features
# 'DESCR'         -> description of the dataset

In [ ]:
# --- SETUP (run this cell) ---
print(dataset['DESCR'])

In [ ]:
# --- SETUP (run this cell) ---
df = pandas.DataFrame(dataset['data'], columns=dataset['feature_names'])
df['disease_progression'] = dataset['target']

In [ ]:
df.describe()

> **Observation:** The means of the features are close to zero and the standard deviations are similar. The data has already been **normalized**.

In [ ]:
seaborn.pairplot(df)

---
### Step 2: Split Into Training and Test Sets

**Why do we split?** We want to evaluate our model on data it has *never seen* during training.  
This tells us how well the model **generalizes** to new observations.

In [ ]:
# --- SETUP (run this cell) ---
from sklearn.model_selection import train_test_split

print("Features shape:", dataset['data'].shape)
print("Target shape:", dataset['target'].shape)

In [ ]:
# TODO: Split the data into training (70%) and test (30%) sets.
# Hint: Use train_test_split(dataset['data'], dataset['target'], test_size=..., random_state=56)
# Store the result in: X_train, X_test, y_train, y_test
# Expected output: 4 arrays (no printed output, just the assignment)


---
### Step 3: Train a Linear Regression Model

**What is linear regression?**  
We fit a model: $\hat{y} = a + b_1 x_1 + b_2 x_2 + \ldots + b_{10} x_{10}$  
where $a$ is the intercept and $b_i$ are the coefficients.

Scikit-learn pattern: **create** a model object → **fit** it on training data.

In [ ]:
from sklearn.linear_model import LinearRegression

# TODO: Create a LinearRegression model and fit it on X_train, y_train
# Hint: model = LinearRegression() then model.fit(...)
# Expected output: LinearRegression() (the fitted model)


In [ ]:
# TODO: Print the model's intercept and coefficients
# Hint: model.intercept_ and model.coef_
# Expected output: one number (intercept) and an array of 10 numbers (coefficients)


---
### Step 4: Evaluate the Model

We use the **$R^2$ score** to evaluate model quality.  
$R^2 = 1$ means perfect prediction; $R^2 = 0$ means the model predicts no better than the mean.

Compare the training score and test score to detect **overfitting**.

In [ ]:
# TODO: Compute and print the R² score on the test set AND the training set
# Hint: model.score(X_test, y_test) and model.score(X_train, y_train)
# Expected output: two numbers, both around 0.5


---
### Step 5: Should We Adjust the Test Set Size?

Let's try different splits and see how the score changes.

> ⚠️ **Warning:** This is shown for illustration only — it is **bad practice** because we are repeatedly peeking at the test data to make decisions.

In [ ]:
# TODO: Loop over different test_size values and record the test R² score each time
# Hint:
#   sizes = [0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
#   scores = []
#   for s in sizes:
#       split -> fit -> score -> append
# Expected output: a list of 10 scores


In [ ]:
# TODO: Plot scores vs. sizes
# Hint: plt.plot(sizes, scores, marker='o')
# Expected output: a line plot showing score fluctuation


> **Takeaway:** A single train/test split gives an unstable estimate. The solution? **Cross-validation.**

---
### Step 6: K-Fold Cross-Validation

**Why cross-validation?** Instead of one random split, we rotate which part of the data is used for testing.  
With $k$-fold, the data is split into $k$ equal parts. Each part takes a turn as the test set while the rest is used for training.

In [ ]:
from sklearn.model_selection import KFold

X = dataset['data']
y = dataset['target']

# TODO: Implement 3-fold cross-validation
# Hint:
#   kf = KFold(n_splits=3)
#   scores = []
#   for train_index, test_index in kf.split(X):
#       split X,y using indices -> fit -> score -> append
#   then print scores and their mean
# Expected output: 3 scores and their average


---
### Step 7: Introduction to Lasso Regression

**Lasso** adds a penalty to the size of the coefficients — it pushes some toward zero, performing **variable selection**.

In [ ]:
from sklearn.linear_model import Lasso

# --- SETUP: fresh split ---
X_train, X_test, y_train, y_test = train_test_split(
    dataset['data'], dataset['target'], test_size=0.3, random_state=56
)

# TODO: Create a Lasso model, fit it, and print its test R² score
# Hint: same pattern as LinearRegression — Lasso() then .fit() then .score()
# Expected output: R² score (likely lower than plain linear regression)


> **Question:** Is the Lasso score better or worse than the linear regression score? Why might that be?

---
## Part 2: California Housing — Sparse Regressions

Now we apply the same workflow to a larger, real-world dataset:  
the **California Housing** dataset (median house values across California districts).

### Step 1: Load and Explore the Data

In [ ]:
# --- SETUP (run this cell) ---
from sklearn.datasets import fetch_california_housing
dataset = fetch_california_housing()
print(dataset['DESCR'])

### Step 2: Split the Data

In [ ]:
# --- SETUP (run this cell) ---
X = dataset['data']
y = dataset['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=58
)

### Step 3: Lasso Regression

> **Note:** Ideally, we should check whether the data needs normalization before fitting regularized models. For this exercise we proceed with default settings.

In [ ]:
# TODO: Fit a Lasso model on the training data and print the R² score on the test set
# Hint: Lasso() -> .fit(X_train, y_train) -> .score(X_test, y_test)
# Expected output: R² score (one number)


### Step 4: Ridge Regression

**Ridge** regression also penalizes large coefficients, but uses an $L_2$ penalty (squared coefficients) instead of $L_1$ (absolute values). It keeps all features but shrinks coefficients.

In [ ]:
from sklearn.linear_model import Ridge

# TODO: Fit a Ridge model on the training data and print the R² score on the test set
# Hint: Ridge() -> .fit(X_train, y_train) -> .score(X_test, y_test)
# Expected output: R² score (one number, likely higher than Lasso)


> **Final question:** Which model performed better on the California Housing dataset — Lasso or Ridge? What might explain the difference?

> ⚠️ **Caveat:** We used the same test set to compare both models. A cleaner approach would use a separate validation set or cross-validation to choose the model.